# 9장. 회귀 분석으로 숫자 예측하기

이 Notebook의 핵심은 높은 점수를 만드는 것이 아니라 다음 순서를 지키는 것입니다.

`예측 시점 → 누수 방지 → 시간 분할 → Train TimeSeriesSplit → 모델 선택 고정 → Final Test → Baseline 비교`

**Final Test 결과를 보고 모델을 고르지 않습니다.**

## 0. 실행 전 확인

Chapter09는 Chapter05의 오류 탐지용 전용 Raw가 아니라 **공통 프로젝트 `data/raw`**를 사용합니다. 먼저 프로젝트 루트에서 다음 명령을 실행해 관계 검증을 통과한 모델링 입력을 준비합니다.

```powershell
python scripts/prepare_ch09_data.py
```

이 명령은 공통 `data/raw`를 전처리하고 FK 관계를 검증한 뒤 `data/processed/*_clean.csv`를 생성합니다.

In [2]:
from pathlib import Path
import sys
import pandas as pd

CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name == 'notebooks' else CURRENT_DIR
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
REPORT_DIR = PROJECT_ROOT / 'reports'

from src.regression import (
    FEATURE_COLUMNS,
    FORBIDDEN_FEATURES,
    TARGET_COLUMN,
    build_feature_audit,
    build_leakage_checklist,
    build_regression_dataset,
    build_regression_validation,
    build_split_summary,
    create_prediction_result,
    cross_validate_regression_models,
    load_regression_source_data,
    make_regression_models,
    run_regression_analysis,
    save_regression_outputs,
    select_diagnostic_model,
    split_model_data_by_time,
    train_and_evaluate_models,
    validate_feature_columns,
)

print('PROJECT_ROOT:', PROJECT_ROOT)
print('PROCESSED_DIR:', PROCESSED_DIR)

PROJECT_ROOT: G:\내 드라이브\dev_GD\llm-data-analysis-study
PROCESSED_DIR: G:\내 드라이브\dev_GD\llm-data-analysis-study\data\processed


## 1. 예측 문제와 예측 시점

- Target: 주문 1건의 주문 상세 금액 합계 `order_total`
- 예측 시점: 주문 메타데이터와 고객의 비식별 특성은 알 수 있지만 주문 상세 수량·단가·금액은 모델에 제공하지 않는 시점
- 목표 계산 재료와 사후 정보, 식별자는 feature에서 제외합니다.

In [3]:
print('Target:', TARGET_COLUMN)
print('Allowed features:', FEATURE_COLUMNS)
print('Forbidden features:', sorted(FORBIDDEN_FEATURES))
validate_feature_columns()
print('Feature leakage gate: PASS')

Target: order_total
Allowed features: ['order_month', 'order_dayofweek', 'age', 'payment_method', 'gender', 'city']
Forbidden features: ['avg_unit_price', 'customer_id', 'item_count', 'line_total', 'order_id', 'order_status', 'order_total', 'product_id', 'quantity', 'total_quantity', 'unit_price']
Feature leakage gate: PASS


## 2. 검증된 전처리 데이터를 불러옵니다

In [4]:
data = load_regression_source_data(PROCESSED_DIR)
for name, frame in data.items():
    print(name, frame.shape)

customers (150, 6)
orders (300, 7)
order_items (764, 6)


## 3. 주문 단위 Target을 만들고 관계를 검증합니다

`line_total = quantity × unit_price`를 확인하고, 주문과 Target은 `one_to_one`, 주문과 고객은 `many_to_one` 관계가 깨지지 않는지 확인합니다.

In [5]:
model_data = build_regression_dataset(
    customers=data['customers'],
    orders=data['orders'],
    order_items=data['order_items'],
)
display(model_data.head())
print('Model rows:', len(model_data))

,order_id,customer_id,order_date,payment_method,order_status,order_month,order_dayofweek,order_total,gender,age,city
0,238,124,2025-07-09,bank_transfer,completed,7,2,361000,M,67,대구
1,79,62,2025-07-11,bank_transfer,refunded,7,4,283000,F,63,광주
2,12,98,2025-07-12,kakao_pay,completed,7,5,444000,F,61,인천
3,169,9,2025-07-12,kakao_pay,completed,7,5,448000,M,69,서울
4,152,62,2025-07-13,naver_pay,refunded,7,6,985000,F,63,광주


Model rows: 300


## 4. Feature Audit을 확인합니다

In [6]:
feature_audit = build_feature_audit()
display(feature_audit)

,column,selected,role,reason
0,order_month,True,allowed_feature,교육용 예측 시점에 사용 가능하다고 가정
1,order_dayofweek,True,allowed_feature,교육용 예측 시점에 사용 가능하다고 가정
2,age,True,allowed_feature,교육용 예측 시점에 사용 가능하다고 가정
3,payment_method,True,allowed_feature,교육용 예측 시점에 사용 가능하다고 가정
4,gender,True,allowed_feature,교육용 예측 시점에 사용 가능하다고 가정
5,city,True,allowed_feature,교육용 예측 시점에 사용 가능하다고 가정
6,avg_unit_price,False,forbidden,주문 상세에서 만든 목표 대리 변수
7,customer_id,False,forbidden,고객 식별자
8,item_count,False,forbidden,주문 상세가 확인된 뒤 계산되는 사후 집계
9,line_total,False,forbidden,목표값을 구성하는 주문 상세 금액


## 5. 시간 순서로 Train / Final Test를 분리합니다

같은 달력 날짜를 양쪽에 나누지 않습니다. Final Test는 모델 선택이 끝날 때까지 사용하지 않습니다.

In [7]:
train_data, test_data = split_model_data_by_time(model_data, test_size=0.2)
split_summary = build_split_summary(train_data, test_data)
display(split_summary)
assert train_data['order_date'].max().normalize() < test_data['order_date'].min().normalize()
print('Chronological split gate: PASS')

,split,rows,ratio_pct,start_date,end_date
0,train,244,81.33,2025-07-09,2026-04-13
1,test,56,18.67,2026-04-14,2026-07-08


Chronological split gate: PASS


In [8]:
X_train = train_data[FEATURE_COLUMNS].copy()
X_test = test_data[FEATURE_COLUMNS].copy()
y_train = train_data[TARGET_COLUMN].copy()
y_test = test_data[TARGET_COLUMN].copy()
print('Train:', X_train.shape, 'Final Test:', X_test.shape)

Train: (244, 6) Final Test: (56, 6)


## 6. Pipeline 후보 모델을 준비합니다

숫자형 결측 대체·표준화와 범주형 결측 대체·One-Hot Encoding은 모델 Pipeline 안에서 각 Train Fold로만 학습됩니다.

In [9]:
models = make_regression_models(random_state=42)
print(list(models))

['Baseline Mean', 'Linear Regression', 'Random Forest']


## 7. Train 기간의 TimeSeriesSplit으로 후보를 비교합니다

여기까지는 Final Test의 `y_test`를 모델 선택에 사용하지 않습니다.

In [10]:
cv_summary = cross_validate_regression_models(
    models=models,
    X_train=X_train,
    y_train=y_train,
)
display(cv_summary)

,model,n_splits,cv_MAE_mean,cv_MAE_std,cv_R2_mean
0,Baseline Mean,5,423004.808743,44153.043466,-0.022821
1,Random Forest,5,437352.803929,26318.793429,-0.109687
2,Linear Regression,5,461386.286905,77792.799324,-0.310517


## 8. 비베이스라인 모델 선택을 고정합니다

Linear Regression과 Random Forest 중 Train CV의 평균 MAE가 더 낮은 모델을 선택합니다. **이 시점까지 Final Test 성능은 보지 않습니다.**

In [11]:
selected_model_name = select_diagnostic_model(cv_summary)
print('Frozen selected model:', selected_model_name)

Frozen selected model: Random Forest


## 9. 이제 Final Test에서 Baseline과 고정 모델만 평가합니다

Final Test 결과가 기대와 다르더라도 같은 Test를 보고 다른 후보로 교체하지 않습니다.

In [12]:
model_comparison, predictions = train_and_evaluate_models(
    models=models,
    selected_model_name=selected_model_name,
    X_train=X_train,
    X_test=X_test,
    y_train=y_train,
    y_test=y_test,
)
display(model_comparison)

,model,selection_role,train_MAE,test_MAE,test_RMSE,test_R2,MAE_improvement_vs_baseline_pct
0,Baseline Mean,baseline,416266.191884,443449.648712,522189.699620,-0.001121,0.00
1,Random Forest,selected_by_train_cv,326438.478132,458819.917234,548190.446723,-0.103298,-3.47


### 지표 읽기

- MAE: 평균적으로 얼마만큼 틀리는가
- RMSE: 큰 오차에 더 민감
- R²: 평균 예측 대비 설명력, 음수가 될 수 있음
- MAE 개선율: Dummy baseline 대비 고정 모델의 Final Test MAE 개선 정도

## 10. 실제값·예측값·잔차를 진단합니다

In [13]:
prediction_result = create_prediction_result(
    test_data=test_data,
    y_test=y_test,
    y_pred=predictions[selected_model_name],
    model_name=selected_model_name,
)
display(prediction_result.head(10))
print('주의: order_id가 포함된 이 표는 내부 진단용입니다.')

,order_id,order_date,actual_order_total,predicted_order_total,residual,abs_error,model
0,51,2026-06-15,1877000,605528.715709,1.271471e+06,1.271471e+06,Random Forest
1,21,2026-06-19,1767000,756889.654706,1.010110e+06,1.010110e+06,Random Forest
2,145,2026-05-27,1863000,861784.708271,1.001215e+06,1.001215e+06,Random Forest
3,96,2026-05-11,1690000,694946.179725,9.950538e+05,9.950538e+05,Random Forest
4,183,2026-05-23,1685000,745836.316118,9.391637e+05,9.391637e+05,Random Forest
5,168,2026-06-11,10000,897415.394314,-8.874154e+05,8.874154e+05,Random Forest
6,15,2026-05-23,1677000,849179.542426,8.278205e+05,8.278205e+05,Random Forest
7,179,2026-07-08,86000,904745.854532,-8.187459e+05,8.187459e+05,Random Forest
8,282,2026-04-24,1566000,758941.292777,8.070587e+05,8.070587e+05,Random Forest
9,73,2026-06-14,72000,871636.144070,-7.996361e+05,7.996361e+05,Random Forest


주의: order_id가 포함된 이 표는 내부 진단용입니다.


## 11. 자동 Validation Evidence를 만듭니다

In [14]:
validation = build_regression_validation(
    train_data=train_data,
    test_data=test_data,
    cv_summary=cv_summary,
    selected_model_name=selected_model_name,
    model_comparison=model_comparison,
)
display(validation)
assert validation['status'].eq('PASS').all()

,check,value,status
0,forbidden_feature_overlap,0,PASS
1,strict_train_before_test,True,PASS
2,selected_model_exists_in_train_cv,True,PASS
3,final_test_contains_baseline,True,PASS
4,final_test_contains_frozen_selected_model,True,PASS
5,test_rows_for_r2,56,PASS


## 12. Evidence·내부 진단·공개 보고서를 구분해 저장합니다

In [15]:
checklist = build_leakage_checklist()
output_paths = save_regression_outputs(
    model_data=model_data,
    split_summary=split_summary,
    feature_audit=feature_audit,
    cv_summary=cv_summary,
    selected_model_name=selected_model_name,
    model_comparison=model_comparison,
    prediction_result=prediction_result,
    validation=validation,
    checklist=checklist,
    report_dir=REPORT_DIR,
)
for name, path in output_paths.items():
    print(name, '->', path)

model_data_internal -> G:\내 드라이브\dev_GD\llm-data-analysis-study\reports\ch09_regression_model_data_internal.csv
split_summary -> G:\내 드라이브\dev_GD\llm-data-analysis-study\reports\ch09_regression_split_summary.csv
feature_audit -> G:\내 드라이브\dev_GD\llm-data-analysis-study\reports\ch09_regression_feature_audit.csv
cv_summary -> G:\내 드라이브\dev_GD\llm-data-analysis-study\reports\ch09_regression_cv_summary.csv
model_comparison -> G:\내 드라이브\dev_GD\llm-data-analysis-study\reports\ch09_regression_model_comparison.csv
predictions_internal -> G:\내 드라이브\dev_GD\llm-data-analysis-study\reports\ch09_regression_predictions_internal.csv
validation -> G:\내 드라이브\dev_GD\llm-data-analysis-study\reports\ch09_regression_validation.csv
checklist -> G:\내 드라이브\dev_GD\llm-data-analysis-study\reports\ch09_regression_checklist.csv
report -> G:\내 드라이브\dev_GD\llm-data-analysis-study\reports\ch09_regression_report.md
actual_vs_predicted_figure -> G:\내 드라이브\dev_GD\llm-data-analysis-study\reports\figures\ch09_actual_vs_p

## 13. 전체 공통 함수를 새 실행으로 다시 확인합니다

셀의 메모리 상태가 아니라 공통 함수가 같은 순서를 재현하는지 확인합니다.

In [16]:
result = run_regression_analysis(
    processed_dir=PROCESSED_DIR,
    report_dir=REPORT_DIR,
    test_size=0.2,
    random_state=42,
)
print('Selected by Train CV:', result['selected_model_name'])
display(result['validation'])

Selected by Train CV: Random Forest


,check,value,status
0,forbidden_feature_overlap,0,PASS
1,strict_train_before_test,True,PASS
2,selected_model_exists_in_train_cv,True,PASS
3,final_test_contains_baseline,True,PASS
4,final_test_contains_frozen_selected_model,True,PASS
5,test_rows_for_r2,56,PASS


## 14. 해석

낮은 성능이나 음수 R²도 숨기지 않습니다. 현재 예측 시점에 허용한 정보가 약하다면 `현재 데이터로는 모델 사용 보류`도 올바른 결론입니다.

LLM이 만든 코드도 같은 예측 시점·누수·Pipeline·Train CV·Final Test 보호 기준으로 검토합니다.

# Chapter 09 답안 양식. 회귀 분석으로 숫자 예측하기

> 이 내용을 `chapter09.ipynb`의 Markdown 셀로 작성합니다.

## 제출 정보
- 이름: 김희철
- GitHub ID: KHeeCheol
- 작성일: 2026-11-02
- 최종 Notebook URL: https://github.com/KHeeCheol/llm-data-analysis-study/blob/main/notebooks/chapter09.ipynb

## 1. 예측 문제 정의
- Target: `order_total` — 주문별 `line_total` 합계
- 예측 단위: 주문 1건(`order_id` 기준)
- 예측 시점: 주문 날짜·결제수단 및 고객의 기본 특성은 확인할 수 있지만 주문 상세의 수량·단가·금액 정보는 모델에 제공되지 않는 시점으로 가정한다.
- 예측 시점에 알 수 있는 정보: `order_month`, `order_dayofweek`, `age`, `payment_method`, `gender`, `city`
- 예측 시점에 알 수 없는 정보: 주문 상세의 `quantity`, `unit_price`, `line_total` 및 최종 주문 금액 `order_total`. 최종 주문 상태 `order_status`도 사후 정보이므로 사용하지 않는다.

### 나의 해석과 판단
주문 금액이 확정되기 전에 이용 가능한 정보만으로 금액을 추정하는 상황을 가정하였다. 이를 통해 실제 예측 시점에 사용할 수 없는 정보를 Feature에서 제외하고 데이터 누수 없이 모델의 예측 성능을 검증하고자 한다. 다만 주문 상세 금액을 이미 알고 있다면 order_total은 직접 계산할 수 있으므로 이번 예측 문제는 회귀분석과 데이터 누수 방지 절차를 학습하기 위한 교육용 설정이다.

---

## 2. Feature Leakage Audit

| feature | 예측 시점 사용 가능 | Target 계산 재료 | 사후 정보/ID | 최종 사용 | 판단 이유 |
|---|---|---|---|---|---|
| `order_month` | 예 | 아니오 | 아니오 | 사용 | 주문 시점의 월 정보 |
| `order_dayofweek` | 예 | 아니오 | 아니오 | 사용 | 주문 시점의 요일 정보 |
| `age` | 예 | 아니오 | 아니오 | 사용 | 사전에 확인 가능한 고객 특성 |
| `payment_method` | 조건부 | 아니오 | 아니오 | 사용 | 결제수단 선택 이후의 예측 시점을 가정 |
| `gender` | 예 | 아니오 | 아니오 | 사용 | 사전에 확인 가능한 고객 특성 |
| `city` | 예 | 아니오 | 아니오 | 사용 | 사전에 확인 가능한 고객 지역 |
| `avg_unit_price` | 아니오 | 간접 관련 | 사후 집계 | 제외 | 주문 상세를 이용한 Target 대리변수 |
| `customer_id` | 예 | 아니오 | 식별자 | 제외 | 고객 식별자로 일반적인 예측 특성에서 제외 |
| `item_count` | 아니오 | 간접 관련 | 사후 집계 | 제외 | 주문 상세 확정 후 계산되는 정보 |
| `line_total` | 아니오 | 직접 관련 | 주문 상세 금액 | 제외 | Target을 구성하는 금액 |
| `order_id` | 예 | 아니오 | 식별자 | 제외 | 주문 식별자로 예측 특성에서 제외 |
| `order_status` | 아니오 | 아니오 | 사후 정보 | 제외 | 예측 시점 이후 확정될 수 있는 주문 상태 |
| `order_total` | 아니오 | Target 자체 | 정답 정보 | 제외 | 모델이 예측해야 하는 값 |
| `product_id` | 아니오 | 아니오 | 주문 상세 식별자 | 제외 | 현재 예측 시점에 주문 상세를 제공하지 않음 |
| `quantity` | 아니오 | 직접 관련 | 주문 상세 정보 | 제외 | Target 계산에 사용되는 수량 |
| `total_quantity` | 아니오 | 간접 관련 | 사후 집계 | 제외 | 주문 상세를 이용한 Target 대리변수 |
| `unit_price` | 아니오 | 직접 관련 | 주문 상세 정보 | 제외 | Target 계산에 사용되는 단가 |

- 금지 feature overlap 개수: 0개
- 가장 위험하다고 판단한 feature: `order_total`, `line_total`, `quantity`, `unit_price`
- 이유: `order_total`은 Target 자체이며 나머지 세 변수는 Target을 계산하는 데 직접 사용되는 정보이다. 이를 Feature에 포함하면 모델이 예측 시점에 알 수 없는 정답 관련 정보를 학습하게 되어 실제 예측 성능이 과대평가될 수 있다.
이번 audit에는 허용 Feature 6개와 금지 Feature 11개가 명확히 구분되어 기본 Leakage Gate를 통과하였다. 다만 `payment_method` 등의 사용 가능 여부는 실제 업무의 예측 시점에 따라 달라질 수 있으므로, 현재 결과는 교육용 예측 시점의 가정에 따른 판단이다.

![Leakage 점검](images/step02_leakage.png)

---

## 3. Target 생성과 관계 검증

### 계산 기준

```text
line_total = quantity × unit_price
order_total = 동일한 order_id에 속한 line_total의 합계
```

- line_total 검증 결과: 주문 상세 764행에서 계산 불일치 0건
- orders ↔ target 관계 검증 결과: `one_to_one` 병합 통과, 주문 300건에 대응하는 Target 생성
- orders → customers 관계 검증 결과: `many_to_one` 병합 통과, 고객 정보 미매칭 없음

### 나의 해석과 판단
주문 상세를 `order_id`별로 집계하여 주문 1건당 하나의 Target을 생성하였다. 최종 모델링 데이터가 주문 수와 동일한 300행으로 확인되어 병합 과정의 누락이나 행 증가 없이 데이터가 구성되었다고 판단하였다.
병합 관계를 검증하지 않으면 주문 중복으로 Target이 잘못 연결되거나 고객 정보가 없는 주문이 누락되어 학습 데이터와 평가 결과가 왜곡될 수 있다. 따라서 모델 학습 전에 Target 계산과 PK/FK 관계 검증이 필요하다.
---

## 4. Train / Final Test 분할

- Train 시작일: 2025-07-09
- Train 종료일: 2026-04-13
- Train 행 수: 244건 (81.33%)
- Final Test 시작일: 2026-04-14
- Final Test 종료일: 2026-07-08
- Final Test 행 수: 56건 (18.67%)
- 같은 달력 날짜 중복 여부: 없음

### 분할 판단
이번 분석은 과거 주문 정보를 바탕으로 이후 주문 금액을 예측하는 문제이므로 무작위 분할보다 시간 순서 분할이 적절하다고 판단하였다. 무작위 분할은 미래 주문이 Train에 포함되어 실제 예측 환경보다 평가 결과가 낙관적으로 나타날 위험이 있다.
따라서 과거 244건을 Train으로 사용하고 이후 56건을 Final Test로 분리하였다. 모델 선택은 Train 내부의 TimeSeriesSplit 결과만으로 수행하며 Final Test는 모델을 고정한 이후 최종 성능 평가에만 사용한다.

![시간 분할](images/step04_split.png)

---

## 5. Train TimeSeriesSplit 후보 비교

| 모델 | CV MAE 평균 | CV MAE 표준편차 | CV R² 평균 | 비고 |
|---|---:|---:|---:|---|
| Baseline Mean | 423,005원 | 44,153원 | -0.0228 | 전체 후보 중 MAE 최저 |
| Linear Regression | 461,386원 | 77,793원 | -0.3105 | 비베이스라인 후보 |
| Random Forest | 437,353원 | 26,319원 | -0.1097 | 비베이스라인 후보 중 우수 |

### Train CV로 선택한 비베이스라인 모델
- Selected Model: Random Forest
- 선택 기준: Train 기간의 5-Fold TimeSeriesSplit에서 비베이스라인 후보 중 `cv_MAE_mean`이 가장 낮은 모델을 선택하였다. Random Forest의 CV MAE는 437,353원으로 Linear Regression의 461,386원보다 낮았다.
- Final Test 결과를 보기 전에 선택했는가: 예

### 나의 해석과 판단
Baseline Mean의 CV MAE가 423,005원으로 가장 낮아 현재 Train 데이터에서는 복잡한 모델의 성능 개선 효과가 확인되지 않았다. 다만 과제의 모델 선택 기준에 따라 비베이스라인 후보 중 Random Forest를 선택하였다.
Random Forest는 CV MAE 평균뿐 아니라 표준편차도 Linear Regression보다 작아 Fold 간 오차 변동이 상대적으로 적었다. CV 평균만으로 판단하면 일부 기간에서 성능이 크게 달라지는 문제를 놓칠 수 있으므로 표준편차도 함께 고려할 필요가 있다.
모든 모델의 CV R² 평균이 음수였다는 점에서 현재 Feature의 예측력에는 한계가 있을 가능성이 있다. 최종 판단은 모델을 고정한 후 Baseline과 동일한 Final Test에서 비교하여 수행한다.
---

## 6. Final Test: Baseline vs Frozen Model

| 모델 | selection role | Train MAE | Test MAE | Test RMSE | Test R² | Baseline 대비 MAE 개선율 |
|---|---|---:|---:|---:|---:|---:|
| Baseline Mean | baseline | 416,266원 | 443,450원 | 522,190원 | -0.0011 | 0.00% |
| Random Forest | selected_by_train_cv | 326,438원 | 458,820원 | 548,190원 | -0.1033 | -3.47% |

![최종 회귀 평가](images/step06_final_metrics.png)

### 결과 관찰
Train CV에서 선택한 Random Forest의 Final Test MAE는 458,820원으로 Baseline Mean의 443,450원보다 약 15,370원 높았다. Baseline 대비 MAE 개선율은 -3.47%였으며 두 모델 모두 Test R²가 음수로 나타났다.

### 나의 해석과 판단
- Frozen Model이 Baseline보다 개선되었는가? 개선되지 않았다. Random Forest는 Train MAE가 더 낮았지만 Final Test에서는 Baseline보다 오차가 컸다. 학습 데이터에 대한 적합도가 실제 미래 데이터의 성능 향상을 보장하지 않음을 확인하였다.
- MAE와 RMSE 차이: Random Forest의 Test RMSE는 548,190원으로 MAE 458,820원보다 높았다. 일부 큰 예측 오차가 존재할 가능성이 있으므로 대표 오차 사례를 추가 확인할 필요가 있다.
- R² 해석: Random Forest의 Test R²는 -0.1033으로 현재 Feature가 Final Test 주문 금액의 변동을 충분히 설명하지 못한 것으로 판단된다.

### 업무·분석적 의미
현재 Random Forest는 주문당 평균 약 45.9만 원의 절대 예측 오차를 보이며 단순 평균 예측보다도 성능이 낮았다. 따라서 실제 주문 금액 예측에 적용하기에는 추가 검증이 필요하다.
특히 주문 상세 정보를 사용할 수 없는 예측 시점에서 현재 Feature만으로 충분한 예측력이 확보되는지 재검토할 필요가 있다. 다만 모델은 Final Test 이전에 고정되었으므로 이번 결과를 근거로 다른 후보로 변경하지 않는다.
---

## 7. 대표 오차 사례

### 사례 1 — 고액 주문 과소예측
- 실제값: 1,877,000원
- 예측값: 약 605,529원
- residual: +1,271,471원
- absolute error: 1,271,471원
- 관찰: 실제 주문 금액이 예측값보다 약 127만 원 높았으며 Final Test에서 가장 큰 절대 오차를 기록하였다.
- 가능한 이유 후보: 현재 Feature에는 주문별 상품 구성, 수량, 단가 정보가 없어 고액 주문의 특성이 충분히 반영되지 않았을 가능성이 있다.
- 추가 확인: 해당 주문의 상품 구성과 금액 분포를 사후적으로 확인하고 고액 주문에서 유사한 과소예측이 반복되는지 분석할 필요가 있다.

### 사례 2 — 소액 주문 과대예측
- 실제값: 10,000원
- 예측값: 약 897,415원
- residual: -887,415원
- absolute error: 887,415원
- 관찰: 실제 주문 금액이 매우 낮았지만 모델은 약 90만 원으로 예측하여 상당한 과대예측이 발생하였다.
- 가능한 이유 후보: 주문 시점과 고객의 기본 특성만으로는 개별 소액 주문을 구분하기 어려웠을 가능성이 있다.
- 추가 확인: 소액 주문의 빈도와 공통 특성을 확인하고 실제 예측 시점에 확보 가능한 추가 정보가 있는지 검토할 필요가 있다.

> 원인을 데이터가 직접 보여 주지 않는다면 사실처럼 단정하지 않습니다.

---

## 8. Validation Evidence

`reports/ch09_regression_validation.csv` 기준으로 작성하세요.

| check | value | status |
|---|---|---|
| forbidden_feature_overlap | 0 | PASS |
| strict_train_before_test | True | PASS |
| selected_model_exists_in_train_cv | True | PASS |
| final_test_contains_baseline | True | PASS |
| final_test_contains_frozen_selected_model | True | PASS |
| test_rows_for_r2 | 56 | PASS |

### FAIL 항목
- 없음 / 있음: 없음
- 있다면 원인과 수정 내용: 해당 없음. Feature Leakage, 시간순 Train/Final Test 분할, Train CV 기반 모델 선택 및 Final Test 평가 조건이 모두 PASS하였다.

---

## 9. Evidence와 공개 범위

### 생성된 Evidence
- [x] `ch09_regression_split_summary.csv`
- [x] `ch09_regression_feature_audit.csv`
- [x] `ch09_regression_cv_summary.csv`
- [x] `ch09_regression_model_comparison.csv`
- [x] `ch09_regression_validation.csv`
- [x] `ch09_regression_checklist.csv`

### Internal 파일
- [x] `ch09_regression_model_data_internal.csv`
- [x] `ch09_regression_predictions_internal.csv`

### 공개 결과
- [x] `ch09_regression_report.md`
- [x] `ch09_actual_vs_predicted.png`
- [x] `ch09_residual_histogram.png`

### 공개 범위 판단
회귀분석의 내부 모델링 및 오차 진단에는 주문별 관계를 추적하기 위해 원본 order_id가 필요하지만 공개 보고서에서는 해당 식별자를 제공할 필요가 없다고 판단하였다.
따라서 원본 주문 식별자가 포함될 수 있는 모델링 데이터와 예측 결과는 Internal 파일로 구분하고 GitHub 공개 저장소에는 업로드하지 않는다. 외부 공개 결과는 식별정보를 제외한 검증 요약, 모델 성능 지표, 보고서 및 그래프로 제한한다. 특히 Random Forest의 Final Test MAE가 Baseline보다 3.47% 높았다는 결과를 그대로 기록하여 모델 성능의 한계도 투명하게 제시한다.
---

## 10. 최종 사용 판단

- [ ] 현재 수준에서 참고용 사용 가능
- [ ] 추가 검증 후 사용 가능
- [x] 현재 데이터로는 사용 보류

### 판단 근거
1. Train CV 결과: Random Forest는 비베이스라인 후보 중 가장 낮은 CV MAE(437,353원)를 기록하였으나 Baseline Mean(423,005원)보다 오차가 높았다. 현재 Feature만으로는 복잡한 모델의 추가적인 예측 효과가 확인되지 않았다.
2. Final Test 결과: Random Forest의 MAE는 458,820원으로 Baseline Mean의 443,450원보다 높았으며 개선율은 -3.47%였다. RMSE는 548,190원, R²는 -0.1033으로 나타나 실제 미래 데이터에 대한 예측 성능이 충분하지 않았다.
3. 검증 및 오차 분석: Feature Leakage와 시간순 분할 등 Validation 6개 항목은 모두 PASS하였다. 그러나 개별 주문에서는 최대 약 127만 원의 절대 오차가 발생하였으며 고액 주문 과소예측과 소액 주문 과대예측이 모두 확인되었다.

### 업무적 위험
주문 금액을 크게 과소예측하거나 과대예측할 경우 재고·예산·판매 계획 등에 부정확한 판단을 유발할 수 있다. 현재 모델은 단순 평균 예측보다 성능이 낮으므로 실제 의사결정에 직접 활용하기 어렵다고 판단하였다.
### 현재 데이터의 한계
현재 Feature는 주문 시점과 고객의 기본 특성으로 제한되어 있어 상품 구성이나 주문별 구매 규모를 충분히 설명하지 못할 수 있다. 또한 실제 주문 상세의 수량과 단가가 확정된 이후에는 `order_total`을 정확히 계산할 수 있으므로, 현재 문제는 실무 예측보다 회귀분석 절차를 학습하는 데 의미가 크다.
### 다음 개선 우선순위
1. 예측 시점에 실제 확보할 수 있는 추가 Feature를 검토하고 Target 계산 재료가 포함되지 않도록 Leakage Audit을 유지한다.
2. 주문 금액 구간별 오차와 시간에 따른 성능 변동을 분석하여 고액·소액 주문에서 큰 오차가 발생하는 원인을 탐색한다.
3. 향후 기간의 판매 금액이나 판매 수량처럼 실제 예측 필요성이 있는 Target을 정의하고 새로운 시간 구간에서 Baseline 대비 성능을 검증한다.

---

## 11. 전체 재실행 확인

실행 명령:

```powershell
python -m scripts.prepare_ch09_data
python -m scripts.run_regression_analysis
```

- 실행 성공 여부: PASS. 공통 `data/raw` 데이터의 전처리부터 FK 검증, 시간순 분할, Train CV, Final Test 및 결과 파일 생성까지 정상적으로 수행되었다.
- Train CV Selected Model: Random Forest. 비베이스라인 후보 중 Train CV MAE가 가장 낮은 모델로 Final Test 평가 이전에 선택하였다.
- Final Validation 전체 PASS 여부: PASS. Feature Leakage, 시간순 분할, Train CV 모델 선택, Final Test 평가 대상 및 테스트 데이터 수에 관한 6개 검증이 모두 통과하였다.
- Notebook 결과와 스크립트 결과의 일관성: Train 244건, Final Test 56건으로 일치하였다. Random Forest의 Test MAE 458,819.92원, RMSE 548,190.45원, R² -0.1033 및 Baseline 대비 MAE 개선율 -3.47%도 동일하게 재현되었다.


---

## 최종 체크

- [x] Target과 예측 시점을 정의했습니다.
- [x] Leakage Audit을 수행했습니다.
- [x] Target 계산과 관계 검증을 확인했습니다.
- [x] Train과 Final Test가 시간 순서로 엄격히 분리되었습니다.
- [x] 전처리가 Pipeline 내부에서 학습됩니다.
- [x] Train TimeSeriesSplit으로 후보를 비교했습니다.
- [x] Final Test 전에 Selected Model을 고정했습니다.
- [x] Final Test에서는 Baseline과 Frozen Model만 비교했습니다.
- [x] MAE, RMSE, R²를 함께 해석했습니다.
- [x] 대표 오차 사례를 관찰과 가설로 구분했습니다.
- [x] Validation Evidence가 모두 PASS입니다.
- [x] Internal 결과와 공개 결과를 구분했습니다.
- [x] 낮은 성능도 숨기지 않았습니다.
- [x] 전체 스크립트를 재실행했습니다.
- [x] 최종 Notebook URL을 제출합니다.


## 15. 완료 체크

- [x] `python scripts/prepare_ch09_data.py`로 공통 모델링 입력을 준비했다.
- [x] Target과 예측 시점을 설명할 수 있다.
- [x] 금지 feature가 입력에 없음을 확인했다.
- [x] 같은 날짜가 Train과 Final Test에 동시에 들어가지 않는다.
- [x] 전처리는 Pipeline 내부에서 학습된다.
- [x] Train TimeSeriesSplit으로 후보를 비교했다.
- [x] Final Test 전에 Selected Model을 고정했다.
- [x] Final Test에서는 Baseline과 Frozen Model만 비교했다.
- [x] MAE, RMSE, R²를 함께 읽었다.
- [x] 내부 식별자 결과와 공개 보고서를 구분했다.
- [x] Validation Evidence가 모두 PASS다.